# DLinear

In [ ]:
%pip install -q mlflow torch scikit-learn pandas matplotlib

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    pass
sys.path.insert(0, os.getcwd())     # src ro dainaxos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import torch
from src.dl import (INPUT_LEN, HORIZON, build_series_matrix, holiday_weight_vector,
                    mean_abs_scale, WindowDataset, DLinear, NBeats, train_model,
                    GlobalForecastPyfunc)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from src.data import load_raw, submission_ids
from src.features import WalmartFeatureBuilder, MARKDOWN_COLS
from src.metrics import wmae, holiday_weights
from src.validation import year_back_split, time_folds, DEV_TRAIN_END, TRAIN_END
from src.tracking import setup_mlflow, REGISTERED_MODEL_NAME

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("MLflow tracking:", setup_mlflow("DLinear_Training"))

In [ ]:
raw = load_raw()
train, test = raw["train"], raw["test"]

tr_idx, va_idx = year_back_split(train["Date"])
dev_train, dev_valid = train.iloc[tr_idx], train.iloc[va_idx]
print(f"train: {train.shape}, test: {test.shape}")
print(f"dev_train: {len(dev_train):,} რიგი (<= {DEV_TRAIN_END.date()}), "
      f"dev_valid: {len(dev_valid):,} რიგი (39 კვირა, შეიცავს Thanksgiving/Christmas/Super Bowl-ს)")

In [ ]:
values, mask, keys, dates = build_series_matrix(train)
hol_w = holiday_weight_vector(dates)
dev_end = int(np.searchsorted(dates, DEV_TRAIN_END, side="right"))
T = len(dates)

def scaled_upto(upto):
    v, m = values[:, :upto], mask[:, :upto]
    s = mean_abs_scale(v, m)
    return (v / s[:, None]).astype(np.float32), s

def year_back_wmae(model, scaled, scales):
    hist = scaled[:, dev_end - INPUT_LEN:dev_end]
    pf = GlobalForecastPyfunc(model, hist, scales, keys, dates[dev_end - 1])
    pred = pf.predict(None, dev_valid)
    return wmae(dev_valid["Weekly_Sales"], pred, dev_valid["IsHoliday"])

def log_epoch(epoch, train_loss, valid_loss):
    mlflow.log_metric("train_loss", train_loss, step=epoch)

with mlflow.start_run(run_name="DLinear_Preprocessing"):
    mlflow.log_params({"grid": "W-FRI", "missing_policy": "0 + mask",
                       "scaling": "per-series mean-abs",
                       "input_len": INPUT_LEN, "horizon": HORIZON,
                       "loss": "masked holiday-weighted MAE (=WMAE)"})
    mlflow.log_metrics({"n_series": len(keys), "n_weeks": T,
                        "observed_share": float(mask.mean())})

scaled_dev, scales_dev = scaled_upto(dev_end)
train_ds_dev = WindowDataset(scaled_dev, mask[:, :dev_end], hol_w[:dev_end])
print(f"{len(keys)} მწკრივი, {T} კვირა | dev window-ები: {len(train_ds_dev)}")

In [ ]:
EPOCHS = 40
best_wmae, best_kernel = None, None
with mlflow.start_run(run_name="DLinear_CV"):
    for kernel in (7, 13, 25):
        with mlflow.start_run(run_name=f"kernel_{kernel}", nested=True):
            torch.manual_seed(RANDOM_STATE)
            model = DLinear(kernel=kernel)
            model = train_model(model, train_ds_dev, epochs=EPOCHS,
                                batch_size=512, lr=1e-3, device=DEVICE,
                                log_fn=log_epoch)
            score = year_back_wmae(model, scaled_dev, scales_dev)
            mlflow.log_params({"kernel": kernel, "epochs": EPOCHS})
            mlflow.log_metric("valid_wmae", score)
            print(f"kernel={kernel}: valid WMAE {score:.0f}")
            if best_wmae is None or score < best_wmae:
                best_wmae, best_kernel = score, kernel

print(f"\nსაუკეთესო kernel: {best_kernel} (WMAE {best_wmae:.0f})")

In [ ]:
scaled_full, scales_full = scaled_upto(T)
train_ds_full = WindowDataset(scaled_full, mask, hol_w)
print("full window-ები:", len(train_ds_full))

with mlflow.start_run(run_name="DLinear_Final") as run:
    torch.manual_seed(RANDOM_STATE)
    model = DLinear(kernel=best_kernel)
    model = train_model(model, train_ds_full, epochs=EPOCHS,
                        batch_size=1024, lr=1e-3, device=DEVICE,
                        log_fn=log_epoch)

    mlflow.log_params({"kernel": best_kernel, "epochs": EPOCHS,
                       "input_len": INPUT_LEN, "horizon": HORIZON})
    mlflow.log_metric("valid_wmae", best_wmae)

    pyfunc_model = GlobalForecastPyfunc(model, scaled_full[:, T - INPUT_LEN:],
                                        scales_full, keys, dates[-1])
    mlflow.pyfunc.log_model(name="model", python_model=pyfunc_model,
                            code_paths=["src"])
    dlinear_final_run_id = run.info.run_id

print("run:", dlinear_final_run_id)

In [ ]:
REGISTER_AS_CHAMPION = False

if REGISTER_AS_CHAMPION:
    from mlflow import MlflowClient
    mv = mlflow.register_model(f"runs:/{dlinear_final_run_id}/model", REGISTERED_MODEL_NAME)
    MlflowClient().set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", mv.version)
    print(f"დარეგისტრირდა: {REGISTERED_MODEL_NAME} v{mv.version} (alias: champion)")